<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система уведомлений</h2>

----

### Вариант задания 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Программа на C# демонстрирует базовые принципы ООП:
1. Инкапсуляция — скрытие данных с помощью свойств.
2. Наследование — создание классов Email, SMS, Push на основе базового класса Notification.
3. Полиморфизм — переопределение метода SendNotification в каждом классе.
4. Абстракция — использование абстрактного класса для задания общего шаблона.

Также показано разделение обязанностей (принцип SRP): 
базовый класс отвечает за общие данные, а наследники — за специфику отправки.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

public interface ISoundable
{
    void MakeSound();
}

public interface ITrackable
{
    void Track();
}

public interface IPrioritizable
{
    void SetPriority(int level);
}

public abstract class Notification
{
    private int _id;
    public int Id
    {
        get { return _id; }
        set
        {
            if (value > 0) _id = value;
            else throw new ArgumentOutOfRangeException("ID должен быть > 0!");
        }
    }

    private string _message;
    public string Message
    {
        get { return _message; }
        set
        {
            if (!string.IsNullOrEmpty(value)) _message = value;
            else throw new ArgumentException("Сообщение не может быть пустым!");
        }
    }

    private string _type;
    public string Type
    {
        get { return _type; }
        set { _type = value; }
    }

    public DateTime CreatedAt { get; set; }

    public static int TotalNotifications { get; private set; } = 0;

    public Notification(int id, string message, string type)
    {
        Id = id;
        Message = message;
        Type = type;
        CreatedAt = DateTime.Now;
        TotalNotifications++;
    }

    public abstract void SendNotification();

    public void DisplayNotification()
    {
        Console.WriteLine($"#{Id} [{Type}] от {CreatedAt:HH:mm:ss}: {Message}");
    }

    public virtual string GetSummary()
    {
        return $"Уведомление #{Id} типа {Type}";
    }

    public bool IsValid()
    {
        return Id > 0 && !string.IsNullOrEmpty(Message);
    }

    public static void ShowTotal()
    {
        Console.WriteLine($"[Статистика] Всего уведомлений: {TotalNotifications}");
    }
}

public class EmailNotification : Notification, ISoundable
{
    private string _emailAddress;
    public string EmailAddress
    {
        get { return _emailAddress; }
        set
        {
            if (value.Contains("@")) _emailAddress = value;
            else throw new ArgumentException("Некорректный email!");
        }
    }

    public string Subject { get; set; }
    public bool HasAttachment { get; set; }

    public EmailNotification(int id, string message, string email, string subject, bool hasAttachment)
        : base(id, message, "Email")
    {
        EmailAddress = email;
        Subject = subject;
        HasAttachment = hasAttachment;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"[Email] → {EmailAddress}");
        Console.WriteLine($"        Тема: {Subject}");
        Console.WriteLine($"        Вложение: {(HasAttachment ? "Да" : "Нет")}");
    }

    public void MakeSound()
    {
        Console.WriteLine("[Email] *звук уведомления письма*");
    }

    public override string GetSummary()
    {
        return $"Email → {EmailAddress} (тема: {Subject})";
    }
}

public class SMSNotification : Notification
{
    private string _phoneNumber;
    public string PhoneNumber
    {
        get { return _phoneNumber; }
        set
        {
            if (value.StartsWith("+") || value.StartsWith("8"))
                _phoneNumber = value;
            else throw new ArgumentException("Некорректный номер!");
        }
    }

    public int CharacterCount { get; set; }
    public string Operator { get; set; }

    public SMSNotification(int id, string message, string phone, string op)
        : base(id, message, "SMS")
    {
        PhoneNumber = phone;
        CharacterCount = message.Length;
        Operator = op;
    }

    public override void SendNotification()
    {
        Console.WriteLine($"[SMS] → {PhoneNumber} ({Operator})");
        Console.WriteLine($"      Длина: {CharacterCount} символов");
    }

    public bool IsWithinLimit()
    {
        return CharacterCount <= 160;
    }

    public override string GetSummary()
    {
        return $"SMS → {PhoneNumber} ({CharacterCount} симв.)";
    }
}

public class UrgentSMS : SMSNotification, IPrioritizable
{
    public int PriorityLevel { get; set; }
    public bool RequiresConfirmation { get; set; }
    public int DeliveryTime { get; set; }

    public UrgentSMS(int id, string message, string phone, string op,
                     int priority, bool requiresConfirm, int deliveryTime)
        : base(id, message, phone, op)
    {
        PriorityLevel = priority;
        RequiresConfirmation = requiresConfirm;
        DeliveryTime = deliveryTime;
        Type = "UrgentSMS";
    }

    public void SetPriority(int level)
    {
        PriorityLevel = level;
        Console.WriteLine($"[UrgentSMS] Приоритет установлен: {level}");
    }

    public override void SendNotification()
    {
        Console.WriteLine($"[URGENT SMS] → {PhoneNumber}");
        Console.WriteLine($"             Приоритет: {PriorityLevel}");
        Console.WriteLine($"             Подтверждение: {(RequiresConfirmation ? "Да" : "Нет")}");
        Console.WriteLine($"             Доставка за: {DeliveryTime} сек.");
    }

    public void ConfirmDelivery()
    {
        Console.WriteLine($"[UrgentSMS] Доставка подтверждена!");
    }

    public override string GetSummary()
    {
        return $"СРОЧНО! SMS → {PhoneNumber} (приоритет {PriorityLevel})";
    }
}

public class PushNotification : Notification, ISoundable, ITrackable, IPrioritizable
{
    private string _platform;
    public string Platform
    {
        get { return _platform; }
        set
        {
            if (value == "iOS" || value == "Android")
                _platform = value;
            else throw new ArgumentException("Только iOS или Android!");
        }
    }

    public string Icon { get; set; }
    public int PriorityLevel { get; set; }
    public string TrackingId { get; set; }

    public PushNotification(int id, string message, string platform)
        : base(id, message, "Push")
    {
        Platform = platform;
        Icon = platform == "iOS" ? "🍎" : "🤖";
        PriorityLevel = 1;
        TrackingId = Guid.NewGuid().ToString().Substring(0, 8);
    }

    public override void SendNotification()
    {
        Console.WriteLine($"{Icon} [Push] → {Platform}");
        Console.WriteLine($"        Приоритет: {PriorityLevel}");
        Console.WriteLine($"        Трекер: {TrackingId}");
    }

    public void MakeSound()
    {
        Console.WriteLine("[Push] *звук push-уведомления*");
    }

    public void Track()
    {
        Console.WriteLine($"[Push] Отслеживание: {TrackingId}");
    }

    public void SetPriority(int level)
    {
        PriorityLevel = level;
    }

    public override string GetSummary()
    {
        return $"Push → {Platform} (трекер: {TrackingId})";
    }
}

Console.WriteLine("--- 1. ПРОСТОЕ НАСЛЕДОВАНИЕ ---");
EmailNotification email = new EmailNotification(
    1, "Ваш заказ отправлен", "client@mail.ru", "Заказ #1234", true
);
email.DisplayNotification();
email.SendNotification();
email.MakeSound();
Console.WriteLine("Сводка: " + email.GetSummary());
Console.WriteLine();

Console.WriteLine("--- 2. СЛОЖНОЕ НАСЛЕДОВАНИЕ ---");
SMSNotification sms = new SMSNotification(
    2, "Код: 5678", "+7-999-123-45-67", "МТС"
);
sms.DisplayNotification();
sms.SendNotification();
Console.WriteLine("Укладывается в лимит? " + sms.IsWithinLimit());
Console.WriteLine();

UrgentSMS urgent = new UrgentSMS(
    3, "СРОЧНО! Проверьте аккаунт", "+7-999-999-99-99", "Билайн",
    5, true, 3
);
urgent.DisplayNotification();
urgent.SendNotification();
urgent.SetPriority(10);
urgent.ConfirmDelivery();
Console.WriteLine("Сводка: " + urgent.GetSummary());
Console.WriteLine();

Console.WriteLine("--- 3. МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ ---");
PushNotification push = new PushNotification(
    4, "У вас новое сообщение!", "Android"
);
push.DisplayNotification();
push.SendNotification();
push.MakeSound();
push.Track();
push.SetPriority(3);
push.DisplayNotification();
Console.WriteLine("Сводка: " + push.GetSummary());
Console.WriteLine();

Console.WriteLine("--- 4. ПОЛИМОРФИЗМ ---");
Notification[] all = { email, sms, urgent, push };
foreach (Notification n in all)
{
    Console.WriteLine("→ " + n.GetSummary());
    n.SendNotification();
    Console.WriteLine();
}

Console.WriteLine("--- 5. РАБОТА С ИНТЕРФЕЙСАМИ ---");
ISoundable[] soundables = { email, push };
Console.WriteLine("Все, кто умеет звучать:");
foreach (ISoundable s in soundables)
{
    s.MakeSound();
}
Console.WriteLine();

IPrioritizable[] priorities = { urgent, push };
Console.WriteLine("Все с приоритетом:");
foreach (IPrioritizable p in priorities)
{
    p.SetPriority(7);
}
Console.WriteLine();

Notification.ShowTotal();

--- 1. ПРОСТОЕ НАСЛЕДОВАНИЕ ---
#1 [Email] от 23:03:09: Ваш заказ отправлен
[Email] → client@mail.ru
        Тема: Заказ #1234
        Вложение: Да
[Email] *звук уведомления письма*
Сводка: Email → client@mail.ru (тема: Заказ #1234)

--- 2. СЛОЖНОЕ НАСЛЕДОВАНИЕ ---
#2 [SMS] от 23:03:09: Код: 5678
[SMS] → +7-999-123-45-67 (МТС)
      Длина: 9 символов
Укладывается в лимит? True

#3 [UrgentSMS] от 23:03:09: СРОЧНО! Проверьте аккаунт
[URGENT SMS] → +7-999-999-99-99
             Приоритет: 5
             Подтверждение: Да
             Доставка за: 3 сек.
[UrgentSMS] Приоритет установлен: 10
[UrgentSMS] Доставка подтверждена!
Сводка: СРОЧНО! SMS → +7-999-999-99-99 (приоритет 10)

--- 3. МНОЖЕСТВЕННОЕ НАСЛЕДОВАНИЕ ---
#4 [Push] от 23:03:09: У вас новое сообщение!
🤖 [Push] → Android
        Приоритет: 1
        Трекер: 2fdfdea7
[Push] *звук push-уведомления*
[Push] Отслеживание: 2fdfdea7
#4 [Push] от 23:03:09: У вас новое сообщение!
Сводка: Push → Android (трекер: 2fdfdea7)

--- 4. ПОЛИМОРФИЗ